# GuidaPlate — XGBoost Risk Classifier v2 (Improved)
## Notebook 04b — Isolated improvement pipeline

**Does not modify** notebook 04 or `models/xgboost_v1.pkl`.

Improvements over v1:
- Interaction features (7 new)
- Cost-sensitive training (MODERATE = 3× sample weight)
- RandomizedSearchCV with `f1_macro` scoring
- SHAP analysis + McNemar vs rule baseline
- Decision gate for saving `models/xgboost_v2.pkl`


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings
import time

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV, learning_curve,
)
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, f1_score, roc_curve, auc,
)
from sklearn.preprocessing import label_binarize
from statsmodels.stats.contingency_tables import mcnemar
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
try:
    plt.style.use('seaborn-v0_8')
except OSError:
    plt.style.use('seaborn')
%matplotlib inline

RANDOM_STATE = 42
TEST_SIZE = 0.2

STAGE_ORDER = ['G2', 'G3a', 'G3b', 'G4']
STAGE_ENCODE = {'G2': 1, 'G3a': 2, 'G3b': 3, 'G4': 4}
RISK_CLASSES = ['LOW', 'MODERATE', 'HIGH']
RISK_ENCODE = {c: i for i, c in enumerate(RISK_CLASSES)}

KDOQI = {
    'G2':  {'potassium': 3500, 'phosphorus': 1000, 'protein_per_kg': 0.8, 'sodium': 2300},
    'G3a': {'potassium': 3000, 'phosphorus': 800,  'protein_per_kg': 0.6, 'sodium': 2300},
    'G3b': {'potassium': 3000, 'phosphorus': 800,  'protein_per_kg': 0.6, 'sodium': 2300},
    'G4':  {'potassium': 2500, 'phosphorus': 700,  'protein_per_kg': 0.55, 'sodium': 2300},
}

def project_root() -> Path:
    p = Path.cwd().resolve()
    if p.name == 'notebooks':
        return p.parent
    if (p / 'data' / 'processed' / 'ckd_cohort_final.csv').exists():
        return p
    if (p.parent / 'data' / 'processed' / 'ckd_cohort_final.csv').exists():
        return p.parent
    return p

ROOT = project_root()
FIG_DIR = ROOT / 'outputs' / 'figures'
STATS_DIR = ROOT / 'outputs' / 'stats'
MODEL_DIR = ROOT / 'models'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Project root: {ROOT}')


## Section 1 — Setup & Data Loading


In [ ]:
cohort = pd.read_csv(ROOT / 'data' / 'processed' / 'ckd_cohort_final.csv')
labels = pd.read_csv(STATS_DIR / '05_risk_labels.csv')

df = cohort.merge(labels[['SEQN', 'risk_label']], on='SEQN', how='inner')
df = df.dropna(subset=['risk_label'])
nutrient_cols = ['potassium', 'phosphorus', 'protein_per_kg', 'sodium']
df = df.dropna(subset=nutrient_cols)

print('=' * 50)
print('DATA LOADED')
print('=' * 50)
print(f'Shape after merge and dropna: {df.shape}')
print()
print('Risk label distribution:')
print(df['risk_label'].value_counts().reindex(RISK_CLASSES))
print()
print('Stage distribution:')
print(df['ckd_stage'].value_counts().reindex(STAGE_ORDER))

y = df['risk_label']


In [ ]:
# DIAGRAM 1 — Class distribution bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x=y, ax=axes[0],
              palette={'LOW': '#22c55e', 'MODERATE': '#f59e0b', 'HIGH': '#ef4444'},
              order=RISK_CLASSES)
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Risk Label')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha='center', va='bottom',
    )

ratios = y.value_counts(normalize=True).reindex(RISK_CLASSES) * 100
sns.barplot(x=ratios.index, y=ratios.values, ax=axes[1],
            palette={'LOW': '#22c55e', 'MODERATE': '#f59e0b', 'HIGH': '#ef4444'})
axes[1].set_title('Class Distribution (%)')
axes[1].set_ylabel('Percentage')
for p in axes[1].patches:
    axes[1].annotate(
        f'{p.get_height():.1f}%',
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha='center', va='bottom',
    )
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_01_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_01_class_dist.png"}')


## Section 2 — Feature Engineering


In [ ]:
def get_ratio(row, nutrient):
    stage = row['ckd_stage']
    if stage not in KDOQI:
        return np.nan
    limit = KDOQI[stage][nutrient]
    if limit == 0:
        return np.nan
    return row[nutrient] / limit

df['potassium_ratio'] = df.apply(lambda r: get_ratio(r, 'potassium'), axis=1)
df['phosphorus_ratio'] = df.apply(lambda r: get_ratio(r, 'phosphorus'), axis=1)
df['protein_ratio'] = df.apply(lambda r: get_ratio(r, 'protein_per_kg'), axis=1)
df['sodium_ratio'] = df.apply(lambda r: get_ratio(r, 'sodium'), axis=1)
df['ckd_stage_encoded'] = df['ckd_stage'].map(STAGE_ENCODE)
df['risk_encoded'] = df['risk_label'].map(RISK_ENCODE)

ratio_cols = ['potassium_ratio', 'phosphorus_ratio', 'protein_ratio', 'sodium_ratio']
df['k_x_p'] = df['potassium_ratio'] * df['phosphorus_ratio']
df['k_x_protein'] = df['potassium_ratio'] * df['protein_ratio']
df['p_x_protein'] = df['phosphorus_ratio'] * df['protein_ratio']
df['total_burden'] = df[ratio_cols].sum(axis=1)
df['max_ratio'] = df[ratio_cols].max(axis=1)
df['nutrients_near_limit'] = df[ratio_cols].apply(
    lambda r: ((r >= 0.8) & (r < 1.0)).sum(), axis=1,
)
df['nutrients_exceeded'] = df[ratio_cols].apply(
    lambda r: (r >= 1.0).sum(), axis=1,
)

FEATURES_V1 = [
    'potassium', 'phosphorus', 'protein_per_kg', 'sodium',
    'potassium_ratio', 'phosphorus_ratio', 'protein_ratio', 'sodium_ratio',
    'ckd_stage_encoded',
]
INTERACTION_FEATURES = [
    'k_x_p', 'k_x_protein', 'p_x_protein', 'total_burden', 'max_ratio',
    'nutrients_near_limit', 'nutrients_exceeded',
]
FEATURES_V2 = FEATURES_V1 + INTERACTION_FEATURES
TARGET = 'risk_encoded'

print('Interaction features added:', INTERACTION_FEATURES)
print(f'Final dataset shape: {df.shape}')
print(f'v1 features: {len(FEATURES_V1)} | v2 features: {len(FEATURES_V2)}')


In [ ]:
# DIAGRAM 2 — Feature correlation heatmap
X_df = df[FEATURES_V2].copy()
fig, ax = plt.subplots(figsize=(14, 10))
corr = X_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0,
    ax=ax, linewidths=0.5,
    annot_kws={'size': 8},
)
ax.set_title('Feature Correlation Matrix\n(Original + Interaction Features)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_02_feature_corr.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_02_feature_corr.png"}')


In [ ]:
# DIAGRAM 3 — Feature distributions by risk class
features_to_plot = [
    'potassium_ratio', 'phosphorus_ratio', 'protein_ratio', 'sodium_ratio',
    'total_burden', 'max_ratio',
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors = {'LOW': '#22c55e', 'MODERATE': '#f59e0b', 'HIGH': '#ef4444'}

for i, feat in enumerate(features_to_plot):
    for label in RISK_CLASSES:
        mask = df['risk_label'] == label
        axes[i].hist(
            df.loc[mask, feat],
            alpha=0.6, bins=30,
            label=label,
            color=colors[label],
        )
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)
    axes[i].axvline(x=1.0, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Feature Distributions by Risk Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_03_feat_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_03_feat_dist.png"}')


## Section 3 — Cost-sensitive Training


In [ ]:
X = df[FEATURES_V2]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)
test_idx = X_test.index
X_train_df = pd.DataFrame(X_train, columns=FEATURES_V2)
X_test_df = pd.DataFrame(X_test, columns=FEATURES_V2)

sample_weight_train = np.ones(len(y_train), dtype=float)
sample_weight_train[y_train.values == RISK_ENCODE['MODERATE']] = 3.0

print(f'Training set size: {len(X_train)}')
print(f'Test set size: {len(X_test)}')
print(f'MODERATE weight count (3×): {(sample_weight_train == 3.0).sum()}')

param_distributions = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
}

base_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    random_state=RANDOM_STATE,
    eval_metric='mlogloss',
)

search = RandomizedSearchCV(
    base_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

print('Starting hyperparameter search (50 iterations, 5-fold CV, f1_macro)...')
start_time = time.time()
search.fit(X_train, y_train, sample_weight=sample_weight_train)
elapsed = time.time() - start_time
print(f'\nSearch completed in {elapsed:.1f} seconds')
print('Best parameters:')
for param, value in search.best_params_.items():
    print(f'  {param}: {value}')
print(f'Best CV F1 macro: {search.best_score_:.4f}')

best_model = xgb.XGBClassifier(
    **search.best_params_,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    verbosity=0,
)
best_model.fit(
    X_train, y_train,
    sample_weight=sample_weight_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)
print('v2 model trained.')


In [ ]:
# DIAGRAM 4 — Learning curves
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train,
    cv=5, scoring='f1_macro',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#0f766e', label='Training score')
ax.fill_between(
    train_sizes,
    train_scores.mean(axis=1) - train_scores.std(axis=1),
    train_scores.mean(axis=1) + train_scores.std(axis=1),
    alpha=0.1, color='#0f766e',
)
ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#f59e0b', label='Validation score')
ax.fill_between(
    train_sizes,
    val_scores.mean(axis=1) - val_scores.std(axis=1),
    val_scores.mean(axis=1) + val_scores.std(axis=1),
    alpha=0.1, color='#f59e0b',
)
ax.set_xlabel('Training Set Size')
ax.set_ylabel('F1 Macro Score')
ax.set_title('XGBoost v2 Learning Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_04_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_04_learning_curves.png"}')


## Section 4 — Evaluation


In [ ]:
v1_path = MODEL_DIR / 'xgboost_v1.pkl'
v1_size = v1_path.stat().st_size
v1_model = joblib.load(v1_path)
print(f'Loaded v1 model: {v1_path} ({v1_size:,} bytes)')

X_test_v1 = X_test_df[FEATURES_V1]
y_pred_v1 = v1_model.predict(X_test_v1)
y_pred_v2 = best_model.predict(X_test_df)
y_prob_v2 = best_model.predict_proba(X_test_df)

cm_labels = ['HIGH', 'LOW', 'MODERATE']
cm_v1 = confusion_matrix(y_test, y_pred_v1, labels=[RISK_ENCODE[c] for c in cm_labels])
cm_v2 = confusion_matrix(y_test, y_pred_v2, labels=[RISK_ENCODE[c] for c in cm_labels])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (cm, title) in zip(axes, [
    (cm_v1, 'XGBoost v1 (Original)'),
    (cm_v2, 'XGBoost v2 (Improved)'),
]):
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=cm_labels, yticklabels=cm_labels, linewidths=0.5,
    )
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_05_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_05_confusion.png"}')

def class_recall(y_true, y_pred, cls_name):
    idx = RISK_ENCODE[cls_name]
    tp = np.sum((y_true == idx) & (y_pred == idx))
    fn = np.sum((y_true == idx) & (y_pred != idx))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def per_class_f1(y_true, y_pred, cls_name):
    idx = RISK_ENCODE[cls_name]
    return f1_score(y_true == idx, y_pred == idx, zero_division=0)

v1_acc = accuracy_score(y_test, y_pred_v1)
v2_acc = accuracy_score(y_test, y_pred_v2)
v1_f1_w = f1_score(y_test, y_pred_v1, average='weighted', zero_division=0)
v2_f1_w = f1_score(y_test, y_pred_v2, average='weighted', zero_division=0)
v1_f1_m = f1_score(y_test, y_pred_v1, average='macro', zero_division=0)
v2_f1_m = f1_score(y_test, y_pred_v2, average='macro', zero_division=0)
v1_auc = roc_auc_score(y_test, v1_model.predict_proba(X_test_v1), multi_class='ovr', average='weighted')
v2_auc = roc_auc_score(y_test, y_prob_v2, multi_class='ovr', average='weighted')

v1_high_sens = class_recall(y_test, y_pred_v1, 'HIGH')
v2_high_sens = class_recall(y_test, y_pred_v2, 'HIGH')
v1_mod_sens = class_recall(y_test, y_pred_v1, 'MODERATE')
v2_mod_sens = class_recall(y_test, y_pred_v2, 'MODERATE')

v1_high_f1 = per_class_f1(y_test, y_pred_v1, 'HIGH')
v2_high_f1 = per_class_f1(y_test, y_pred_v2, 'HIGH')
v1_low_f1 = per_class_f1(y_test, y_pred_v1, 'LOW')
v2_low_f1 = per_class_f1(y_test, y_pred_v2, 'LOW')
v1_mod_f1 = per_class_f1(y_test, y_pred_v1, 'MODERATE')
v2_mod_f1 = per_class_f1(y_test, y_pred_v2, 'MODERATE')

print('\nClassification report — v2:')
print(classification_report(y_test, y_pred_v2, target_names=RISK_CLASSES, zero_division=0))


In [ ]:
classes = ['HIGH', 'LOW', 'MODERATE']
y_test_bin = label_binarize(y_test, classes=[RISK_ENCODE[c] for c in classes])

fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#ef4444', '#22c55e', '#f59e0b']

for i, (cls, color) in enumerate(zip(classes, colors_roc)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_v2[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('XGBoost v2 ROC Curves by Class')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_06_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_06_roc.png"}')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3)
width = 0.35
bars1 = ax.bar(x - width / 2, [v1_high_f1, v1_low_f1, v1_mod_f1], width,
               label='v1 Original', color='#94a3b8')
bars2 = ax.bar(x + width / 2, [v2_high_f1, v2_low_f1, v2_mod_f1], width,
               label='v2 Improved', color='#0f766e')
ax.set_xticks(x)
ax.set_xticklabels(['HIGH', 'LOW', 'MODERATE'])
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1: XGBoost v1 vs v2')
ax.legend()
ax.set_ylim(0, 1.1)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_07_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_07_f1_comparison.png"}')


## Section 5 — SHAP Analysis


In [ ]:
feature_names = FEATURES_V2
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_df)

if isinstance(shap_values, list):
    shap_list = shap_values
else:
    shap_arr = np.asarray(shap_values)
    if shap_arr.ndim == 3:
        shap_list = [shap_arr[:, :, i] for i in range(shap_arr.shape[2])]
    else:
        shap_list = [shap_arr]

classes = ['HIGH', 'LOW', 'MODERATE']
high_idx = classes.index('HIGH')
mod_idx = classes.index('MODERATE')

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_list[high_idx], X_test_df,
    feature_names=feature_names,
    plot_type='beeswarm', show=False, max_display=15,
)
plt.title('SHAP Feature Importance\n(HIGH Risk Class)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_08_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_08_shap_beeswarm.png"}')

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_list[high_idx], X_test_df,
    feature_names=feature_names,
    plot_type='bar', show=False, max_display=15,
)
plt.title('Mean |SHAP| Feature Importance\n(HIGH Risk Class)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_09_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_09_shap_bar.png"}')

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_list[mod_idx], X_test_df,
    feature_names=feature_names,
    plot_type='bar', show=False, max_display=15,
)
plt.title('Mean |SHAP| Feature Importance\n(MODERATE Risk Class)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_10_shap_moderate.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_10_shap_moderate.png"}')


## Section 6 — McNemar Test


In [ ]:
def assign_risk_label(row):
    stage = row['ckd_stage']
    if stage not in KDOQI:
        return None
    limits = KDOQI[stage]
    exceeded = 0
    if pd.notna(row['potassium']) and row['potassium'] > limits['potassium']:
        exceeded += 1
    if pd.notna(row['phosphorus']) and row['phosphorus'] > limits['phosphorus']:
        exceeded += 1
    if pd.notna(row['protein_per_kg']) and row['protein_per_kg'] > limits['protein_per_kg']:
        exceeded += 1
    if pd.notna(row['sodium']) and row['sodium'] > limits['sodium']:
        exceeded += 1
    if exceeded >= 2:
        return 'HIGH'
    if exceeded == 1:
        return 'MODERATE'
    return 'LOW'

test_df = df.loc[test_idx].copy()
y_rule = test_df.apply(assign_risk_label, axis=1).map(RISK_ENCODE).values

v2_correct = (y_pred_v2 == y_test.values)
rule_correct = (y_rule == y_test.values)

b = int(np.sum(rule_correct & ~v2_correct))
c = int(np.sum(v2_correct & ~rule_correct))
n00 = int(np.sum(v2_correct & rule_correct))
n01 = c
n10 = b
n11 = int(np.sum(~v2_correct & ~rule_correct))

print('=' * 50)
print("McNEMAR TEST — v2 vs Rule Baseline")
print('=' * 50)
print(f'b (rule correct, v2 wrong): {b}')
print(f'c (v2 correct, rule wrong):   {c}')
if b + c > 0:
    result = mcnemar(np.array([[b, c], [c, b]]), exact=True)
    p = float(result.pvalue)
else:
    p = 1.0
print(f'p-value: {p:.4f}')
print('\nCompare to original v1 McNemar reference: b=2, c=0, p=0.50')

fig, ax = plt.subplots(figsize=(6, 4))
contingency = np.array([[n00, n01], [n10, n11]])
sns.heatmap(
    contingency, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Baseline Correct', 'Baseline Wrong'],
    yticklabels=['V2 Correct', 'V2 Wrong'], linewidths=0.5,
)
ax.set_title(f'McNemar Contingency Table\nb={b}, c={c}, p={p:.4f}', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v2_11_mcnemar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v2_11_mcnemar.png"}')


## Section 7 — Final Comparison + Decision Gate


In [ ]:
ORIGINAL = {
    'accuracy': 0.9932,
    'f1_weighted': 0.9933,
    'f1_macro': v1_f1_m,
    'auc': 1.0,
    'high_sensitivity': 0.9951,
    'mod_sensitivity': v1_mod_sens,
    'mcnemar_p': 0.50,
}

print('=' * 60)
print('COMPARISON TABLE')
print('=' * 60)
print(f"{'Metric':<16} | {'v1 Original':<12} | {'v2 Improved':<12}")
print('-' * 60)
print(f"{'Accuracy':<16} | {ORIGINAL['accuracy']*100:>10.2f}% | {v2_acc*100:>10.2f}%")
print(f"{'F1 weighted':<16} | {ORIGINAL['f1_weighted']:>12.4f} | {v2_f1_w:>12.4f}")
print(f"{'F1 macro':<16} | {ORIGINAL['f1_macro']:>12.4f} | {v2_f1_m:>12.4f}")
print(f"{'AUC':<16} | {ORIGINAL['auc']:>12.4f} | {v2_auc:>12.4f}")
print(f"{'HIGH sens':<16} | {ORIGINAL['high_sensitivity']*100:>10.2f}% | {v2_high_sens*100:>10.2f}%")
print(f"{'MOD sens':<16} | {ORIGINAL['mod_sensitivity']*100:>10.2f}% | {v2_mod_sens*100:>10.2f}%")
print(f"{'McNemar p':<16} | {ORIGINAL['mcnemar_p']:>12.2f} | {p:>12.4f}")
print('=' * 60)

original_mod_sensitivity = ORIGINAL['mod_sensitivity']

if (v2_acc >= 0.95 and v2_high_sens >= 0.95 and v2_mod_sens > original_mod_sensitivity):
    joblib.dump(best_model, MODEL_DIR / 'xgboost_v2.pkl')
    decision = '✅ V2 saved — improves MODERATE'
elif (v2_mod_sens - original_mod_sensitivity > 0.20 and v2_acc >= 0.92):
    joblib.dump(best_model, MODEL_DIR / 'xgboost_v2.pkl')
    decision = '⚠ TRADE-OFF saved — review manually'
else:
    decision = '❌ V2 did not improve — keeping v1'

print(decision)

metrics_out = pd.DataFrame([{
    'model': 'XGBoost v2',
    'accuracy': round(v2_acc, 4),
    'f1_weighted': round(v2_f1_w, 4),
    'f1_macro': round(v2_f1_m, 4),
    'auc_roc': round(v2_auc, 4),
    'high_sensitivity': round(v2_high_sens, 4),
    'mod_sensitivity': round(v2_mod_sens, 4),
    'mcnemar_b': b,
    'mcnemar_c': c,
    'mcnemar_p': round(p, 4),
    'decision': decision,
    'n_features': len(FEATURES_V2),
    'training_samples': len(X_train),
    'test_samples': len(X_test),
}])
metrics_path = STATS_DIR / '09_xgboost_v2_metrics.csv'
metrics_out.to_csv(metrics_path, index=False)
print(f'\nSaved metrics: {metrics_path}')
print('\nAll figures (xgb_v2_01 … xgb_v2_11) in outputs/figures/')
print('NOTEBOOK 04b COMPLETE')
